# Movie Recommender System — TMDB Dataset

A **content-based** recommender that builds a tag vector for each movie
(overview + genres + keywords + top cast + director) and ranks movies by
cosine similarity.

**Dataset**: [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata) (Kaggle)

**Output artifacts** (saved to `model/`):
- `movie_list.pkl` — processed DataFrame (movie_id, title, tags)
- `similarity.pkl` — 4803×4803 cosine similarity matrix

> ⚠️ `similarity.pkl` is ~95 MB. Either track it with **Git LFS** or add it to `.gitignore` and instruct users to regenerate it by running this notebook.

## 1. Imports & Setup

In [ ]:
import ast
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('model', exist_ok=True)
print('Libraries loaded.')

## 2. Load & Merge Data

In [ ]:
# Adjust paths if running locally instead of on Kaggle
MOVIES_PATH  = 'data/tmdb_5000_movies.csv'
CREDITS_PATH = 'data/tmdb_5000_credits.csv'

movies  = pd.read_csv(MOVIES_PATH)
credits = pd.read_csv(CREDITS_PATH)

print(f'Movies : {movies.shape}   Credits: {credits.shape}')

# credits has: movie_id, title, cast, crew
# movies  has: id, title, …
# Merge on 'title' (both datasets share it; renaming avoids id collision)
movies = movies.merge(credits, on='title')
print(f'After merge: {movies.shape}')
movies.head(2)

## 3. Select Relevant Columns

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.head(2)

## 4. Feature Engineering — Parsing JSON Strings

In [ ]:
def parse_names(text: str) -> list[str]:
    """Extract all 'name' values from a JSON-like string of dicts."""
    return [item['name'] for item in ast.literal_eval(text)]


def top3_cast(text: str) -> list[str]:
    """Return the top-3 cast members (first billed)."""
    return [item['name'] for item in ast.literal_eval(text)[:3]]


def fetch_director(text: str) -> list[str]:
    """Return the director(s) from the crew list."""
    return [
        item['name']
        for item in ast.literal_eval(text)
        if item.get('job') == 'Director'
    ]


def collapse_spaces(names: list[str]) -> list[str]:
    """Remove spaces so 'Sam Mendes' → 'SamMendes' (prevents token split)."""
    return [name.replace(' ', '') for name in names]


# Drop rows where any key column is missing before applying parsers
movies.dropna(inplace=True)
print(f'Rows after dropping NaN: {len(movies)}')

movies['genres']   = movies['genres'].apply(parse_names).apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(parse_names).apply(collapse_spaces)
movies['cast']     = movies['cast'].apply(top3_cast).apply(collapse_spaces)
movies['crew']     = movies['crew'].apply(fetch_director).apply(collapse_spaces)
movies['overview'] = movies['overview'].apply(str.split)

movies.head(2)

## 5. Build Tag Column

In [ ]:
movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)

new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda tokens: ' '.join(tokens).lower())
print(new_df.shape)
new_df.head(3)

## 6. Vectorisation & Cosine Similarity

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
print(f'Vocabulary size: {len(cv.vocabulary_)}  |  Matrix shape: {vectors.shape}')

similarity = cosine_similarity(vectors)
print(f'Similarity matrix shape: {similarity.shape}')

## 7. Sanity Check — Recommend Function

In [ ]:
def recommend(movie: str, top_n: int = 5) -> list[str]:
    """Return the top-N most similar movies."""
    matches = new_df[new_df['title'] == movie]
    if matches.empty:
        return [f"'{movie}' not found in dataset."]
    idx = matches.index[0]
    distances = sorted(
        enumerate(similarity[idx]), key=lambda x: x[1], reverse=True
    )
    return [new_df.iloc[i[0]].title for i in distances[1 : top_n + 1]]


print(recommend('Avatar'))
print(recommend('The Dark Knight'))
print(recommend('Gandhi'))

## 8. Persist Artefacts

In [ ]:
with open('model/movie_list.pkl', 'wb') as f:
    pickle.dump(new_df, f)

with open('model/similarity.pkl', 'wb') as f:
    pickle.dump(similarity, f)

print('Artefacts saved to model/')